# L22 — (s, S) Inventory Scenario Comparison and Policy Search

**Module**: M06 | **Chapter**: 8 | **Lecture**: L22

## Learning Objectives
By the end of this notebook you will be able to:
1. Run a grid search over (s, S) parameters using 30 replications per policy.
2. Construct confidence intervals for each policy's expected cost.
3. Use common random numbers (CRN) to reduce variance in policy comparisons.
4. Frame the policy search as a precursor to the RL approach in Module M12.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.stats import t as t_dist
from itertools import product

## 1. Recap: The (s, S) Simulation from L21

We use the `SSInventory` model from the `simdes` package to keep notebook cells concise.

In [ ]:
from simdes.models.inventory import SSInventory

# Quick single-replication sanity check
model = SSInventory(reorder_point=10, order_up_to=50, demand_rate=3.0, sim_time=365.0, seed=0)
result = model.run()
print("Single replication (s=10, order_up_to=50):")
for k, v in result.items():
    print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k:25s}: {v}")

## 2. Multi-Replication Cost Estimation

In [ ]:
def estimate_policy_cost(s: int, S: int, n_reps: int = 30,
                         demand_rate: float = 3.0,
                         sim_time: float = 365.0) -> dict:
    """Run n_reps replications and return mean cost with 95% CI."""
    costs = []
    for rep in range(n_reps):
        m = SSInventory(reorder_point=s, order_up_to=S, demand_rate=demand_rate,
                        sim_time=sim_time, seed=rep)
        r = m.run()
        costs.append(r.get('avg_total_cost', 0))

    costs = np.array(costs)
    mean  = costs.mean()
    se    = costs.std() / np.sqrt(n_reps)
    hw    = t_dist.ppf(0.975, df=n_reps - 1) * se
    return {'s': s, 'S': S, 'mean': mean, 'ci_lo': mean-hw, 'ci_hi': mean+hw,
            'std': costs.std(), 'n_reps': n_reps}


# Demonstrate on three policies
policies_demo = [(5, 40), (10, 50), (20, 60)]
print(f"{'(s, S)':>10s}  {'Mean cost':>12s}  {'95% CI':>22s}")
print('-' * 50)
for s, S in policies_demo:
    r = estimate_policy_cost(s, S, n_reps=30)
    print(f"({s:>2d},{S:>3d}):  {r['mean']:>12.4f}  "
          f"[{r['ci_lo']:>8.4f}, {r['ci_hi']:>8.4f}]")

## 3. Grid Search Over (s, S) Parameter Space

In [ ]:
# Grid search: s ∈ {5,10,15,20,25}, S ∈ {35,45,55,65}
# Constraint: S > s
s_values = [5, 10, 15, 20, 25]
S_values = [35, 45, 55, 65]
N_REPS   = 30

results = []
print("Running grid search (this may take ~60 seconds)...")
for s, S in product(s_values, S_values):
    if S <= s:
        continue
    r = estimate_policy_cost(s, S, n_reps=N_REPS)
    results.append(r)

grid_df = pd.DataFrame(results).sort_values('mean')
print(f"\nTop 5 policies by mean daily cost:")
print(grid_df.head(5)[['s','S','mean','ci_lo','ci_hi']].to_string(index=False, float_format='{:.4f}'.format))
print(f"\nWorst 5 policies:")
print(grid_df.tail(5)[['s','S','mean','ci_lo','ci_hi']].to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Heatmap of mean daily cost
pivot = grid_df.pivot(index='s', columns='S', values='mean')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Mean daily cost ($/day)')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Order-up-to level S')
ax.set_ylabel('Reorder point s')
ax.set_title(f'(s, S) Policy Grid Search — Mean Daily Cost ({N_REPS} reps each)')

# Mark minimum
best = grid_df.iloc[0]
best_row = list(pivot.index).index(best['s'])
best_col = list(pivot.columns).index(best['S'])
ax.plot(best_col, best_row, 'g*', markersize=15, label=f'Best: (s={best["s"]}, order_up_to={best["S"]})')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Common Random Numbers for Sharper Comparisons

In [ ]:
def crn_policy_comparison(policy_a: tuple, policy_b: tuple,
                           n_reps: int = 30,
                           demand_rate: float = 3.0,
                           sim_time: float = 365.0) -> dict:
    """
    Compare two (s, S) policies using Common Random Numbers (CRN).
    Both policies use the same seed sequence → correlated errors cancel.
    """
    diffs = []
    for rep in range(n_reps):
        # SAME seed for both policies → CRN
        rA = SSInventory(*policy_a, demand_rate=demand_rate,
                          sim_time=sim_time, seed=rep).run()
        rB = SSInventory(*policy_b, demand_rate=demand_rate,
                          sim_time=sim_time, seed=rep).run()
        cost_key = 'avg_total_cost'
        scale    = 1.0
        diffs.append(rA[cost_key] - rB[cost_key])

    diffs = np.array(diffs)
    mean  = diffs.mean()
    se    = diffs.std() / np.sqrt(n_reps)
    hw    = t_dist.ppf(0.975, df=n_reps - 1) * se
    return {'mean_diff': mean, 'ci_lo': mean-hw, 'ci_hi': mean+hw,
            'std_diff': diffs.std(),
            'significant': not (mean-hw <= 0 <= mean+hw)}


# Compare best vs. second-best
top2 = grid_df.head(2)
pA = (int(top2.iloc[0]['s']), int(top2.iloc[0]['S']))
pB = (int(top2.iloc[1]['s']), int(top2.iloc[1]['S']))

crn_result = crn_policy_comparison(pA, pB, n_reps=50)
print(f"CRN comparison: policy A = (s={pA[0]}, order_up_to={pA[1]})  vs  policy B = (s={pB[0]}, order_up_to={pB[1]})")
print(f"  Mean difference (A - B): {crn_result['mean_diff']:.4f} $/day")
print(f"  95% CI: [{crn_result['ci_lo']:.4f}, {crn_result['ci_hi']:.4f}]")
print(f"  Statistically significant? {'Yes' if crn_result['significant'] else 'No (cannot distinguish these two policies)'}")

## 5. Bridge to RL (Module M12)

The grid search above is exhaustive — it evaluates all (s, S) combinations.  
With a large state space or continuous parameters, this is infeasible.  
In Module M12, an RL agent will **learn** the optimal policy by interacting with the simulation, using reward = −daily_cost.

In [ ]:
print("Grid search summary:")
print(f"  Policies evaluated: {len(grid_df)}")
print(f"  Total simulation-years: {len(grid_df) * N_REPS:.0f}")
print()
print("RL approach (Module M12):")
print("  State:   (inventory_level, pending_order: bool)")
print("  Action:  when to order and how much (or: s and S parameters)")
print("  Reward:  -daily_cost  (holding + ordering + shortage)")
print("  Algorithm: Q-learning or PPO on the simdes.envs.InventoryEnv")
print()
print("The RL agent does not evaluate all policies explicitly.")
print("It explores and refines through experience — more efficient for large spaces.")
print()
best = grid_df.iloc[0]
print(f"Grid search optimal: (s={best['s']:.0f}, order_up_to={best['S']:.0f}), cost=${best['mean']:.4f}/day")
print("(RL should converge to a similar or better policy after training.)")

---
## Try It Yourself

1. **Cost breakdown**: Modify the grid search to track holding cost, ordering cost, and shortage cost separately for each policy. Which cost component drives the optimal s upward? Which drives S upward?

2. **Sequential elimination**: Implement a simplified ranking-and-selection procedure: (a) run 10 pilot reps per policy; (b) eliminate all policies whose CI overlaps with the best from above; (c) run 30 more reps only on the survivors. How many replications does this save compared to always running 40 reps per policy?

3. **Sensitivity to lead time**: Fix (s=10, S=50) and vary mean lead time from 1 to 15 days (7 values). For each, run 30 replications and plot mean daily cost. At what lead time does (s=10, S=50) become suboptimal relative to (s=15, S=55)? Use CRN for the comparison.